# Homework 01 : Agentic RAG

In [28]:
!uv add gitsource OpenAI

Resolved 129 packages in 1ms
Checked 119 packages in 2ms


## Loading the LLM Zoomcamp Lessons .md files

In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [2]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

print(f"Q1 : Loaded {len(documents)} lessons.") 

Q1 : Loaded 72 lessons.


In [3]:
documents[0]

{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

In [4]:
documents[0].keys()

dict_keys(['content', 'filename'])

## Index the loaded documents with MinSearch 

In [5]:
from minsearch import Index

index = Index(
    # search by content
    text_fields=['content'],
    # filter by filename
    keyword_fields=['filename']
)

index.fit(documents)

In [7]:
query1 = "How does the agentic loop keep calling the model until it stops?"

search_results = index.search(
    query1,
    num_results=5
)

print(f"Q2 : File name of the First Result ==> {search_results[0]['filename']}")

Q2 : File name of the First Result ==> 01-agentic-rag/lessons/14-agentic-loop.md


## Building RAG Assistant

In [25]:
# get the rag helper file
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py

--2026-07-09 17:14:46--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.111.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2134 (2.1K) [text/plain]
Saving to: ‘rag_helper.py.2’

rag_helper.py.2     100%[===================>]   2.08K  --.-KB/s    in 0s      

2026-07-09 17:14:48 (8.35 MB/s) - ‘rag_helper.py.2’ saved [2134/2134]



Note : modified the rag_helper to work with course lesson data instead of faq data.

In [8]:
import os

from dotenv import load_dotenv
load_dotenv()
from rag_helper import RAGBase
from openai import OpenAI
client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)
model = "openai/gpt-oss-20b"

assistant = RAGBase(
    index=index,
    llm_client=client,
    model= model
)

query2 = "How does the agentic loop keep calling the model until it stops?"
results = assistant.rag(query2)

In [10]:
print(f"Q3 : {results['input_tokens']} input (prompt) tokens sent to the model for this request")

Q3 : 7194 input (prompt) tokens sent to the model for this request


## Chunking

In [12]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

print(f"Q4 : {len(chunks)} splitted from documents")

Q4 : 295 splitted from documents


## RAG with chunking

In [13]:
from minsearch import Index

chunked_index = Index(
    # search by content
    text_fields=['content'],
    # filter by filename
    keyword_fields=['filename']
)

chunked_index.fit(chunks)

In [15]:
assistant = RAGBase(
    index=chunked_index,
    llm_client=client,
    model= model
)

results = assistant.rag(query2)

In [16]:
print(f"Q5 : {results['input_tokens']} input (prompt) tokens sent to the model for this request")

Q5 : 2377 input (prompt) tokens sent to the model for this request


In [19]:
print(f"{round(7194/2377)}x fewer")

3x fewer


## Turning it into an agent

In [20]:
!uv add toyaikit

Resolved 136 packages in 4.28s                                       
Prepared 2 packages in 1.52s                                             
Installed 7 packages in 19ms                                
 + anthropic==0.116.0
 + docstring-parser==0.18.0
 + genai-prices==0.0.70
 + httpcore2==2.3.0
 + httpx2==2.3.0
 + toyaikit==0.0.11
 + truststore==0.10.4


In [23]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [22]:
INSTRUCTIONS = """
You're a course teaching assistant. Answer the student's question using the search tool. 
Make multiple searches with different keywords before answering.
"""
query = "How does the agentic loop work, and how is it different from plain RAG?"

In [27]:
def search(query):

    """
    Search the chunked lessons for entries matching the given query.
    """
    return chunked_index.search(
        query,
        num_results=5
    )

In [28]:
agent_tools = Tools()
agent_tools.add_tool(search)
print(agent_tools.get_tools())

[{'type': 'function', 'name': 'search', 'description': 'Search the chunked lessons for entries matching the given query.', 'parameters': {'type': 'object', 'properties': {'query': {'type': 'string', 'description': 'query parameter'}}, 'required': ['query'], 'additionalProperties': False}}]


In [32]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt = INSTRUCTIONS,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(
        model=model,
        client=client)
)

In [34]:
result   = runner.loop(
    prompt = query,
    callback=callback
)

-> Response received


-> Response received


-> Response received


-> Response received


/home/hsu/Documents/Learning/llm-zoomcamp-hw/.venv/lib/python3.10/site-packages/toyaikit/chat/runners.py:283: UnknownModelWarning: No pricing data for model 'openai/gpt-oss-20b'. Register it with PricingConfig.register_model(...) to get cost calculations.
  cost_info = self.pricing_config.calculate_cost(
